# Act 1 — DNS: getting users to the right endpoint

Every request your application handles starts with a DNS lookup. The browser, the mobile app, the SDK — all of them turn a hostname into an IP before they can connect to anything. The decision about *which* IP they get back is the first lever you have to control where traffic lands.

AWS gives you **Route 53** as the authoritative DNS that does that resolution. The hosted zone is the container for your records; the routing policies decide which record value to return; the health checks gate the answer behind whether the target is alive. This act walks each of those — records, hosted zones, the Alias extension that fills a gap in plain DNS, eight routing policies, health checks, and the resolver that handles hybrid DNS across the AWS-to-on-prem boundary.

## Three edge services, one job

Route 53 is AWS's authoritative DNS — it answers name lookups for the domains you host and decides which endpoint a user's request lands on. CloudFront is AWS's CDN, sitting at hundreds of edge locations to cache content close to users and absorb load off the origin. Global Accelerator is the third piece — it doesn't cache anything, it gets non-HTTP traffic onto AWS's private backbone as early as possible and routes it across AWS infrastructure rather than the public internet. Three services, three different layers of the path between a user and your app, all answering some version of "get the user to the right endpoint, fast".

## DNS and Record Types

DNS resolves a hostname to an address. The browser asks a recursive resolver — its ISP's, or a public one like `1.1.1.1` — which walks the hierarchy: root, then the TLD, then the authoritative name servers for the domain. Route 53 is the authoritative tier for any hosted zone it manages. The resolver caches the answer for the **TTL** (time to live) the record carries and re-asks when it expires.

TTL is the lever you adjust when planning failover. A low TTL — say 60 seconds — means DNS changes propagate fast; a high TTL means fewer queries (cheaper) but slower cutover. Set TTL based on how quickly DNS would need to follow a failure.

| Record | Maps |
|---|---|
| **A** | hostname → IPv4 |
| **AAAA** | hostname → IPv6 |
| **CNAME** | hostname → another hostname |
| **NS** | hosted zone → its authoritative name servers |
| **MX** | mail routing — priority + mail server |
| **TXT** | arbitrary text (domain verification, SPF, DKIM) |
| **SRV** | service location with priority and port |
| **CAA** | which certificate authorities can issue certs for the domain |
| **PTR** | reverse lookup — IP → hostname |

One CNAME constraint matters: **a CNAME cannot live at the zone apex** — you cannot point `example.com` at `myapp.elb.amazonaws.com` with a CNAME. You can do it for `www.example.com`, just not the root. This is the gap that Route 53 **Alias** records fill, and the reason Alias records exist as a separate concept.

## Hosted Zones

A **hosted zone** is the container for the DNS records of a domain. Two kinds.

A **public hosted zone** answers queries from the internet — for any domain you serve externally. A **private hosted zone** answers queries only from VPCs you associate with it, so internal hostnames like `db.internal` resolve to private IPs. Both cost about fifty cents per zone per month. Private hosted zones need the associated VPC to have both `enableDnsSupport` and `enableDnsHostnames` on — the same toggle pair from the VPC chapter.

Route 53 is also a **domain registrar**. You can register a new domain through it directly, and Route 53 auto-creates the public hosted zone and wires the registrar's NS records to point at it. If the domain is registered elsewhere — GoDaddy, Namecheap, anywhere — you can still use Route 53 as the DNS provider: create the hosted zone, copy the four NS records AWS assigns, and paste them into the registrar's name-server settings. The registrar still owns the domain; Route 53 just answers the queries.

**DNSSEC** is available for public hosted zones — Route 53 signs responses with a private key, and validating resolvers can detect tampered answers. Cheap insurance against DNS spoofing if your audience runs validating resolvers.

## Alias Records

An Alias record is a Route 53 extension that points a hostname directly at an AWS resource — not at another hostname. It's a feature of the *answer* logic, not a standard DNS record type. From the client side it looks like an A (or AAAA) record returning an IP; from your side, you don't have to hard-code an IP that might change.

| | CNAME | Alias |
|---|---|---|
| Target | any hostname | AWS resource (specific list) |
| Zone apex | not allowed | **allowed** |
| Query cost | charged | **free** |
| TTL | you set it | managed by AWS |
| Health check integration | no | yes |

Valid Alias targets — the list to memorise — are Elastic Load Balancers (ALB, NLB, GLB, CLB), CloudFront distributions, API Gateway, S3 static-website endpoints, Elastic Beanstalk environments, VPC interface endpoints, Global Accelerator, and other records in the same hosted zone. **EC2 DNS names and RDS endpoints are not on the list** — those still need CNAMEs (and so cannot sit at the zone apex).

## Routing Policies

Routing policies decide *which* record value Route 53 returns when multiple records share the same name. There are eight, and most non-trivial designs combine two or three.

**Simple.** One record per name. If you list multiple values on a single record, Route 53 returns them all and the client picks one. No health checks, no awareness of failure. Right when there's only one target.

**Weighted.** Multiple records under the same name, each with a weight (0–255). Route 53 distributes responses in proportion to the weights — 90 and 10 gives a roughly 90/10 split. Setting a weight to zero halts traffic to that record. Health-check aware — unhealthy weighted records are excluded from the pool. The canonical use is canary deployments and A/B tests, where you shift a small slice of traffic to a new version before going all-in.

**Latency.** Multiple records, one per region. Route 53 returns the record for the region with the lowest measured latency *from the resolver doing the lookup*. Important nuance: it's based on AWS's latency measurements, not geographic distance. A user in Paris might be routed to `us-east-1` if that path happens to be faster than `eu-west-3`. Use it for multi-region active-active when "fastest wins" is the rule.

**Failover.** Active-passive. One record is marked PRIMARY (with a health check attached), one SECONDARY. While the primary's health check passes, Route 53 returns it; the moment it fails, traffic shifts to the secondary. The cleanest pattern for a hot-standby DR site.

**Geolocation.** Routes by the *user's* geographic location — continent, country, or US state. Unlike latency, the decision is strict: a European user gets the European endpoint even if a US endpoint would be faster. Create a default record for traffic that doesn't match any rule. Used for data sovereignty ("EU users stay on EU infrastructure"), localised content, and country-level blocking.

**Geoproximity.** Also location-based, but adds a **bias** dial that expands or shrinks a region's coverage area. Positive bias makes a region absorb more traffic; negative bias makes it shed traffic. The use case is gradual traffic migration between regions — slide the bias instead of flipping records. Requires Route 53 **Traffic Flow**, the visual editor for composite policies.

**IP-based.** Routes by the client's IP address — you supply a list of CIDR blocks and the endpoint each maps to. Used when you know specific things about specific networks: corporate offices to an internal endpoint, an ISP's range to a peered region, etc.

**Multi-value answer.** Returns up to eight randomly-chosen records, but unlike Simple it filters out unhealthy records via health checks. Not a load balancer — it's still DNS, and clients pick from the returned list — but it gives you health-aware DNS-level distribution without standing up an ELB.

| Policy | Decides by | Health checks |
|---|---|---|
| Simple | nothing — single answer | no |
| Weighted | configured weights | yes |
| Latency | measured AWS latency | yes |
| Failover | primary health | **required** |
| Geolocation | user location | yes |
| Geoproximity | location + bias | yes |
| IP-based | client CIDR | yes |
| Multi-value | random + healthy | yes |

## Health Checks

Route 53's health checkers are a fleet on the public internet that probe an endpoint from many locations and decide healthy or unhealthy by majority — typically three of eighteen-plus probers must agree before status flips. Three flavours.

**Endpoint** — direct HTTP/HTTPS/TCP probe to an IP or hostname, on whatever path you specify. Passes on a `2xx` or `3xx` response within four seconds. Optional **string matching** checks the first 5120 bytes of the body for a specific string — useful when the application returns 200 even while broken.

**Calculated** — a logical combination (AND, OR, NOT) of other health checks. Use it when "the service is up" depends on several components being up.

**CloudWatch alarm** — health is read off a CloudWatch alarm. This is the only way to monitor a **private resource**, because Route 53's probers are on the public internet and can't reach private IPs. The pattern is: emit a metric for the resource, set an alarm on it, point a Route 53 health check at the alarm.

Tuning knobs: probe interval is 30 seconds standard or 10 seconds fast (more expensive); failure threshold is the number of consecutive failed probes before status flips. Health checks are also where you wire failure notifications — set a CloudWatch alarm on the `HealthCheckStatus` metric and hang an SNS topic off it.

## Route 53 Resolver and Hybrid DNS

Every VPC has a built-in DNS resolver at the base CIDR plus two — `10.0.0.2` in a `10.0.0.0/16` VPC. That's the **Route 53 Resolver**. It handles AWS service endpoints, public DNS lookups, and any private hosted zones associated with the VPC.

For a hybrid network you usually need queries to flow both ways across the boundary.

| Endpoint | Direction | Purpose |
|---|---|---|
| **Inbound** | on-prem → AWS | on-prem servers resolve names in a private hosted zone |
| **Outbound** | AWS → on-prem | EC2 instances resolve on-prem names |

Each endpoint is a set of ENIs in your subnets — one per AZ for HA. Inbound endpoints accept queries on port 53 from on-prem DNS servers. Outbound endpoints work with **Resolver Rules** — "forward queries for `corp.internal` to the on-prem DNS server at `192.168.1.53`" — so EC2 instances resolve internal corporate names transparently. Resolver rules can be shared across accounts via Resource Access Manager, which is how a multi-account network uses one set of inbound/outbound endpoints from a single networking-hub account.

In [ ]:
import boto3, uuid

r53 = boto3.client("route53")

# Public hosted zone
zone = r53.create_hosted_zone(
    Name="example.com",
    CallerReference=str(uuid.uuid4()),
    HostedZoneConfig={"PrivateZone": False},
)
zone_id = zone["HostedZone"]["Id"].split("/")[-1]

# Alias at the zone apex pointing at an ALB — CNAME would not be allowed here
r53.change_resource_record_sets(
    HostedZoneId=zone_id,
    ChangeBatch={"Changes": [{
        "Action": "CREATE",
        "ResourceRecordSet": {
            "Name": "example.com",
            "Type": "A",
            "AliasTarget": {
                "HostedZoneId": "Z35SXDOTRQ7X7K",      # ALB's regional hosted zone
                "DNSName": "my-alb-1234.us-east-1.elb.amazonaws.com",
                "EvaluateTargetHealth": True,
            },
        },
    }]},
)

# Weighted records — canary 90/10 between v1 and v2
for sid, ip, weight in [("v1", "10.0.1.10", 90), ("v2", "10.0.2.10", 10)]:
    r53.change_resource_record_sets(
        HostedZoneId=zone_id,
        ChangeBatch={"Changes": [{
            "Action": "CREATE",
            "ResourceRecordSet": {
                "Name": "app.example.com", "Type": "A",
                "SetIdentifier": sid, "Weight": weight,
                "TTL": 60, "ResourceRecords": [{"Value": ip}],
            },
        }]},
    )

# Health check used by a Failover record set — primary is monitored, secondary takes over on failure
r53.create_health_check(
    CallerReference=str(uuid.uuid4()),
    HealthCheckConfig={
        "FullyQualifiedDomainName": "api.example.com",
        "Port": 443, "Type": "HTTPS", "ResourcePath": "/health",
        "RequestInterval": 30, "FailureThreshold": 3, "EnableSNI": True,
    },
)

# Act 2 — CloudFront: caching content close to users

DNS picks the endpoint, but the endpoint might be on another continent. The bytes still have to travel from the user's machine to your origin and back, and the speed of light is fixed.

**CloudFront** is AWS's content delivery network. It sits at six-hundred-plus edge locations close to users and caches your content so most requests are answered locally — the user never traverses the public internet back to your origin. Beyond raw caching, CloudFront terminates TLS at the edge, includes free DDoS protection through Shield Standard, integrates with WAF for application-layer attacks, and brings HTTP/2 and HTTP/3 regardless of what the origin speaks.

This act walks how CloudFront actually works — what it serves from, how the cache key is built, what the security knobs look like, the two flavours of edge functions, and how price classes let you trade global coverage for cost.

## CloudFront

CloudFront is AWS's content delivery network. The job is the standard CDN job: cache content close to the user so most requests are answered from the edge instead of dragging across the internet to the origin. AWS runs the network from 600-plus **edge locations** (Points of Presence) in 100-plus cities, with a tier of larger **regional edge caches** sitting between the edge and the origin to absorb cache misses without hitting origin every time.

What CloudFront actually does in a request:

1. The user's DNS resolves a CloudFront distribution name (or your CNAME/Alias pointing at it) to the nearest edge.
2. The edge checks its cache for the requested object.
3. **Cache hit** — the object is returned immediately from the edge.
4. **Cache miss** — the edge fetches from the regional edge cache; on miss there, from the origin over the AWS backbone. The response is cached on the way back, then returned.

Beyond raw caching, CloudFront brings: **TLS termination** at the edge (much closer than your origin can be), **AWS Shield Standard** included free against L3/L4 DDoS, **WAF** attachment for L7 protections, and **HTTP/2 and HTTP/3** support regardless of what the origin speaks.

## Origins and Origin Access Control

An **origin** is where CloudFront fetches content on a cache miss. Origins can be S3 buckets, S3 static-website endpoints, ALBs, EC2 with a public IP, or any HTTP server anywhere on the internet. A single distribution can have multiple origins, with **cache behaviours** routing different path patterns to different origins.

For an S3 origin the recommended pattern is **Origin Access Control (OAC)** — keep the bucket completely private, then let only CloudFront read from it. CloudFront signs its origin requests with SigV4 and the bucket policy allows the CloudFront service principal scoped to the specific distribution. The user never reaches S3 directly; the bucket has no public access, no presigned-URL leakage to worry about.

OAC replaced the older **Origin Access Identity (OAI)**. OAI still works but is on its way out — OAC supports SSE-KMS encrypted buckets, all S3 regions including newer ones, and dynamic methods like PUT and DELETE. Default to OAC on every new distribution.

**Origin Groups** wrap a primary and secondary origin for origin-level failover. If the primary returns a configured set of HTTP error codes (5xx, certain 4xx) or times out, CloudFront retries the secondary automatically. The classic use is primary S3 bucket in `us-east-1`, secondary S3 bucket in `eu-west-1` with cross-region replication keeping them in sync. The user sees a single distribution; CloudFront handles the failover invisibly.

## Cache Behaviours, TTL, and Cache Keys

A **cache behaviour** is a routing rule on a distribution: a path pattern, the origin it maps to, and the caching settings for that path. Distributions have a default behaviour (`*`) and any number of more specific ones evaluated in precedence order.

| Path | Origin | TTL | Notes |
|---|---|---|---|
| `/api/*` | ALB | 0 | dynamic — don't cache |
| `/images/*` | S3 | 1 day | static |
| `*` (default) | S3 | 1 hour | everything else |

Per behaviour you also configure the viewer protocol policy (HTTP only, HTTPS only, or redirect-HTTP-to-HTTPS), allowed HTTP methods, edge functions attached, and the **cache policy** that defines the cache key.

**TTL.** CloudFront respects `Cache-Control` and `Expires` from the origin, falling back to the behaviour's defaults if none are present. Default TTL is 24 hours, minimum 0 (always revalidate with origin), maximum effectively unbounded. Minimum TTL of 0 still allows objects to be cached at the edge; it just forces a revalidation on each request.

**Invalidations.** Manually evict cached objects by path pattern. Invalidations cost money per path after a small monthly free allowance, and they take a few minutes to fan out across all edge locations. The pattern to avoid invalidations is **versioned filenames** — deploy `app-v2.js` instead of overwriting `app.js`. The new URL is a fresh cache key, so it bypasses any cached copy of the old URL automatically.

**Cache key.** By default the cache key is just the path. **Cache policies** let you add query strings, headers, or cookies to the key — adding `Accept-Language` lets you serve localised content, but every distinct value becomes a separate cache entry. More items in the key = more correctness, lower **cache hit ratio**, more origin load. Include in the key only what actually changes the response.

Sitting alongside cache policies are **origin request policies**, which control what gets forwarded to the origin independent of the cache key. You can forward an `Authorization` header to the origin for per-user logic while keeping it out of the cache key, so all users still share the same cached response.

## CloudFront Security

**Certificates.** CloudFront provides `*.cloudfront.net` with a default cert. For a custom domain you attach an **ACM** certificate — and the certificate must live in **us-east-1** regardless of where your origin or users are, because CloudFront reads its certs from the global control plane there. This catches people out regularly.

**Viewer protocol policy** controls user-to-CloudFront: HTTP only, HTTPS only, or redirect-to-HTTPS. **Origin protocol policy** controls CloudFront-to-origin: HTTP, HTTPS, or match-viewer. The two are independent — encrypt at least the viewer leg, ideally both.

**Geo restriction.** Block or allow countries at the CloudFront layer by IP geolocation. Allowlist mode permits only listed countries; blocklist mode denies them. Blocked requests get a 403. This runs before any cache lookup.

**Signed URLs and signed cookies** grant time-bounded access to private content — paid downloads, premium streams, anything you don't want world-readable even from the CDN. A **signed URL** is the right tool for one specific object; a **signed cookie** covers many objects (an entire video catalogue, say) without you generating a separate URL per file. Both carry an expiry timestamp and a signature CloudFront validates at the edge.

**WAF.** Attaching a WAF web ACL to a distribution adds L7 protection: rate limiting per IP, managed rule groups for common attacks (SQL injection, XSS), allow/block lists, request-body inspection. Combined with **Shield Standard** (free, automatic on every CloudFront distribution) it's the default front line; **Shield Advanced** adds a 24×7 response team and bill-spike protection during attacks.

**Field-level encryption** encrypts specific POST fields — credit card numbers, social security numbers — at the edge using a public key you provide. Only a backend with the matching private key can decrypt them, so the data stays encrypted as it traverses CloudFront and any intermediate hops in your application stack. Niche but important for PCI-style workloads.

## Edge Functions: CloudFront Functions vs Lambda@Edge

CloudFront can run your code at the edge to manipulate requests and responses. Two flavours, very different scale and capability profiles.

| | CloudFront Functions | Lambda@Edge |
|---|---|---|
| Runtime | JavaScript (constrained) | Node.js, Python |
| Trigger points | viewer request, viewer response only | all four — viewer + origin, request + response |
| Max execution | 1 ms | 5 sec viewer / 30 sec origin |
| Memory | 2 MB | 128 MB – 10 GB |
| Network access | no | yes |
| Cost | ~6× cheaper per invocation | higher |
| Scale | millions of req/sec | thousands of req/sec |

**CloudFront Functions** are for the trivially cheap stuff that has to run on every single request: rewriting URLs, normalising cache keys (lowercasing a query string), injecting security headers, simple JWT validation that can be done with a public key embedded in the function. One millisecond of CPU and no network access is the deal — within that envelope they're essentially free at any scale.

**Lambda@Edge** is full Lambda with all the runtime power that implies — call DynamoDB to personalise a response, fetch from an external API, do complex auth against an IdP, choose a different origin per request. The cost is higher per invocation and the latency budget is real, but you get all four trigger points: **viewer request/response** (run on every viewer interaction, cache hit or miss) and **origin request/response** (only on cache misses, so much lower invocation count for the same workload).

The decision is mechanical: if it fits in 1 ms of constrained JavaScript with no network, use Functions. Otherwise Lambda@Edge.

## Price Classes

CloudFront's per-GB cost varies by edge region — Europe and North America are cheapest; Asia-Pacific and South America are more expensive. **Price classes** let you opt out of the pricier regions to save money, at the cost of users in those regions being served from the next-nearest included edge (slightly higher latency).

| Class | Included edges |
|---|---|
| **All** | every edge worldwide |
| **200** | all except South America, Australia, New Zealand |
| **100** | North America and Europe only |

Pick All for genuinely global low-latency requirements; 100 for cost-sensitive apps whose user base is mostly North America and Europe. Price class doesn't affect *who can reach* the content — South American users can still load a Price Class 100 distribution; they're just served from a North American edge.

## CloudFront vs S3 Cross-Region Replication

These come up together because both reduce latency for distant users, but they solve different problems.

| | CloudFront | S3 Cross-Region Replication |
|---|---|---|
| What it does | caches objects at 600+ edges | copies objects to another bucket in another region |
| Freshness | TTL-based; stale until invalidated or expired | near real-time |
| Coverage | automatically global | only the regions you replicate to |
| Writes | read-only at the edge | the destination bucket is fully writable |
| Use case | global content delivery, websites, video | DR, low-latency reads in specific regions, compliance residency |

CloudFront is the right answer when the content is the same everywhere and you want it cached everywhere automatically. Replication is the right answer when you need the data itself living in another region — for failover, for regional read patterns, or for legal residency rules. They combine well: replicate to a backup region and use an **origin group** so CloudFront fails over between buckets if the primary returns errors.

In [ ]:
import boto3, uuid

cf = boto3.client("cloudfront")

# OAC for a private S3 bucket
oac = cf.create_origin_access_control(
    OriginAccessControlConfig={
        "Name": "site-oac",
        "SigningProtocol": "sigv4",
        "SigningBehavior": "always",
        "OriginAccessControlOriginType": "s3",
    },
)["OriginAccessControl"]["Id"]

# Distribution with the S3 origin locked down to that OAC
cf.create_distribution(DistributionConfig={
    "CallerReference": str(uuid.uuid4()),
    "Comment": "static site",
    "Enabled": True,
    "DefaultRootObject": "index.html",
    "Origins": {"Quantity": 1, "Items": [{
        "Id": "s3-origin",
        "DomainName": "my-site-bucket.s3.us-east-1.amazonaws.com",
        "S3OriginConfig": {"OriginAccessIdentity": ""},   # empty when using OAC
        "OriginAccessControlId": oac,
    }]},
    "DefaultCacheBehavior": {
        "TargetOriginId": "s3-origin",
        "ViewerProtocolPolicy": "redirect-to-https",
        "AllowedMethods": {"Quantity": 2, "Items": ["GET", "HEAD"],
                            "CachedMethods": {"Quantity": 2, "Items": ["GET", "HEAD"]}},
        "CachePolicyId": "658327ea-f89d-4fab-a63d-7e88639e58f6",  # managed CachingOptimized
        "Compress": True,
    },
    "PriceClass": "PriceClass_100",
    "HttpVersion": "http2and3",
    "ViewerCertificate": {"CloudFrontDefaultCertificate": True},  # custom domain needs an ACM cert in us-east-1
})

# Invalidation — only when versioned filenames aren't an option, paths cost money
cf.create_invalidation(
    DistributionId="E1ABC2DEFGHIJK",
    InvalidationBatch={
        "Paths": {"Quantity": 2, "Items": ["/index.html", "/static/*"]},
        "CallerReference": str(uuid.uuid4()),
    },
)

# Act 3 — Global Accelerator: getting non-HTTP traffic onto the backbone

CloudFront only helps when the protocol is HTTP. What about a game server speaking custom TCP? A VoIP service on UDP? An IoT fleet using a proprietary protocol? Those still benefit from getting onto AWS's private backbone as early as possible, but caching is not the answer.

**Global Accelerator** is the third piece. It does not cache anything. What it does is give you two static Anycast IP addresses that route the user to AWS's backbone at the nearest edge location, and then carry the traffic across AWS infrastructure to the application. Two static IPs that never change, network-layer failover in seconds, and a private path across AWS's backbone — for any TCP or UDP workload.

## Global Accelerator

Global Accelerator is the third leg of the edge story. Instead of caching, it gets user traffic onto AWS's private backbone at the nearest edge location and carries it across AWS infrastructure to the application — less variability, fewer hops, more predictable latency than the public internet path.

The user-facing surface is **two static Anycast IPv4 addresses** (you can also bring your own IPs). Anycast means the same IP is announced from many edge locations, and the user's request lands at whichever edge is closest. From the edge onward, AWS routes the traffic across its backbone to the healthy endpoint with the lowest latency. Two static IPs that never change is a powerful property — clients can hardcode them, firewalls can allowlist them, and failover happens at the network layer without waiting on any DNS TTL.

**Endpoints** Global Accelerator can route to: ALBs, NLBs, EC2 instances, and Elastic IPs. Endpoints are organised into **endpoint groups**, one per region, each with a **traffic dial** (0–100%) that controls what share of the listener's traffic the group is eligible for — turn it to zero to drain a region during a deployment or incident. Within a group, **endpoint weights** distribute traffic and built-in health checks remove unhealthy endpoints automatically.

**Client IP preservation** matters: with ALB and EC2 endpoints, the original client IP is preserved through Global Accelerator, so your application sees real source IPs (useful for security groups, geo logic, logging). With NLB endpoints it depends on configuration. This is one of the practical reasons to put Global Accelerator in front of a regional ELB instead of routing users straight to the ELB over DNS.

In [ ]:
import boto3

ga = boto3.client("globalaccelerator", region_name="us-east-1")   # GA's API lives in us-east-1 only

acc = ga.create_accelerator(Name="my-app", IpAddressType="IPV4", Enabled=True)["Accelerator"]
acc_arn = acc["AcceleratorArn"]
static_ips = [ip["IpAddress"] for ip in acc["IpSets"][0]["IpAddresses"]]   # the two Anycast IPs

listener = ga.create_listener(
    AcceleratorArn=acc_arn,
    Protocol="TCP",
    PortRanges=[{"FromPort": 443, "ToPort": 443}],
    ClientAffinity="SOURCE_IP",     # same client → same endpoint (sticky)
)["Listener"]["ListenerArn"]

# Endpoint group for us-east-1 with one ALB — another region would add a parallel group
ga.create_endpoint_group(
    ListenerArn=listener,
    EndpointGroupRegion="us-east-1",
    TrafficDialPercentage=100,
    EndpointConfigurations=[{
        "EndpointId": "arn:aws:elasticloadbalancing:us-east-1:111122223333:loadbalancer/app/my-alb/abc",
        "Weight": 100,
        "ClientIPPreservationEnabled": True,
    }],
)

# Act 4 — Picking the right edge service

All three services live at the edge, but they answer different questions. The decision matrix below sorts the choices by layer — DNS resolution, application-layer caching, or network-layer backbone routing — so you can pick the right service quickly once you frame the requirement correctly.

## Picking the Right Edge Service

The three services in this notebook all sit "at the edge", but they answer different questions.

| Need | Service |
|---|---|
| Resolve a hostname; control which endpoint receives traffic by weight, geography, latency, or health | **Route 53** |
| Cache HTTP/HTTPS content close to users; serve static sites, media, websites with high cache-hit ratio | **CloudFront** |
| Get non-HTTP (TCP/UDP) traffic onto AWS's backbone fast, with static IPs and near-instant failover | **Global Accelerator** |

A few sharper splits.

**CloudFront vs Global Accelerator.** CloudFront caches; Global Accelerator doesn't. CloudFront is HTTP/HTTPS/WebSocket only; Global Accelerator handles any TCP or UDP, which is why it's the standard answer for gaming, VoIP, IoT, and any API that can't ride on HTTP. CloudFront's failover is DNS-speed (minutes); Global Accelerator's is network-speed (seconds). Static IPs belong to Global Accelerator — CloudFront's IPs are DNS-managed and change.

**Route 53 latency routing vs Global Accelerator.** Both get global users to the regional endpoint with the lowest latency, but the mechanism is different. Route 53 does it via DNS — the user picks an endpoint at lookup time and is then on the public internet. Global Accelerator does it via Anycast at the network layer — the user lands on AWS's backbone at the nearest edge regardless of DNS. Global Accelerator fails over faster and the path runs over AWS's network, but it costs more than DNS and only fits the endpoint types it supports.

**CloudFront vs S3 Cross-Region Replication** (repeated because it bites people): CloudFront caches the same data everywhere; Replication copies the data to specific regions. They are not substitutes; they often work together.

The framing for picking is always the same — "what layer is the work happening at": DNS, application caching, or backbone routing. The service falls out of that answer.